# CricketMind — results

The three results the paper reports, and the tables it prints. Every number is
measured here and written straight into `paper/CricketMind_ICCIT/tables/`; none
is typed into the LaTeX by hand.

| | Result | Writes |
|---|---|---|
| A | Execution profile — the cost model and its bounds | `tab_execution_profile.tex` |
| B | Query-conditioned routing — keyword vs LLM | three tables |
| C | Boundary of the Faith Check | prose only |

Nothing is defined here either: see the `cricketmind.experiments` package.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
from cricketmind import (AGENT_REGISTRY, CLIP, MAX_ACTIVE_AGENTS, MAX_REGENS,
                         MAX_RETRIES, PROJECT_ROOT, QUERY, Settings, agent_name,
                         new_state)
from cricketmind.experiments import tables as T
from cricketmind.experiments.boundary import R3_AGENT, run_boundary_cases
from cricketmind.experiments.profile import (EXPECTED_COSTS, SCENARIOS,
                                             WORST_AGENT_CALLS, WORST_STEPS,
                                             predicted_steps, profile,
                                             record_run, run_all_scenarios,
                                             run_scenario)
from cricketmind.experiments.routing_eval import (PAPER_FLOW_STAGES,
                                                  ROUTING_PROMPTS, compare,
                                                  load_or_run_llm, record_flows,
                                                  route_all)
print("tables ->", T.TABLES_DIR.relative_to(PROJECT_ROOT))

## A. Execution profile

Node bodies are deterministic, so repeating a query reproduces its execution
exactly. We therefore characterise the cost across *situations* rather than
across repetitions.

LangGraph executes in **supersteps** — rounds in which every scheduled node runs
concurrently. `record_run` reads them from the framework's debug event stream,
so the counts are observed, not asserted.

In [ ]:
cost_rows = run_all_scenarios()

head = f"{'scenario':42s} {'r':>2s} {'g':>2s} {'steps':>6s} {'nodes':>6s} {'agents':>7s}"
print(head); print("-" * len(head))
for r in cost_rows:
    print(f"{r['scenario']:42s} {r['retries']:2d} {r['regens']:2d} "
          f"{r['steps']:6d} {r['node_runs']:6d} {r['agent_calls']:7d}")

for r in cost_rows:
    got = (r["steps"], r["node_runs"], r["agent_calls"])
    assert got == EXPECTED_COSTS[r["scenario"]], f"{r['scenario']}: {got}"
print(f"\nOK: all {len(cost_rows)} scenarios cost what they cost before")

### The cost model

Each event has a fixed superstep cost that can be read off the topology:

$$S = S_0 + 5r + 3g + d,\qquad S_\text{degrade} = 4 + 5r + 1$$

with $S_0 = 12$ with a clip and $9$ without. Nothing below runs the graph — it
is arithmetic over the drawn edges, checked against the measured rows.

In [ ]:
print(f"{'scenario':42s} {'measured':>8s} {'formula':>8s}")
print("-" * 60)
for sc, r in zip(SCENARIOS, cost_rows):
    f = predicted_steps(has_clip=sc["clip"] is not None, retries=r["retries"],
                        regens=r["regens"], dropped=r["dropped"], degraded=r["degraded"])
    print(f"{r['scenario']:42s} {r['steps']:8d} {f:8d}" + ("" if f == r["steps"] else "  <-- MISMATCH"))
    assert f == r["steps"]
print(f"\nOK: the formula matches all {len(cost_rows)} measured runs")

In [ ]:
print(f"limits            MAX_RETRIES={MAX_RETRIES}  MAX_REGENS={MAX_REGENS}  "
      f"MAX_ACTIVE_AGENTS={MAX_ACTIVE_AGENTS}")
print(f"worst steps       {WORST_STEPS}")
print(f"worst agent calls {WORST_AGENT_CALLS}")
print(f"longest run seen  {max(r['steps'] for r in cost_rows)}")
print(f"most calls seen   {max(r['agent_calls'] for r in cost_rows)}")

assert all(r["steps"] <= WORST_STEPS for r in cost_rows)
assert all(r["agent_calls"] <= WORST_AGENT_CALLS for r in cost_rows)
assert max(r["steps"] for r in cost_rows) == WORST_STEPS
print(f"\nOK: no run exceeds the bounds, and the worst case reaches {WORST_STEPS} exactly")

In [ ]:
print(T.write_cost_table(cost_rows))

### The execution-trace figure

Left out of the paper to fit six pages — Table I and Eq. (1) carry the result —
but it is the clearest picture of a correction loop actually happening, so it
belongs in slides and the thesis.

In [ ]:
from cricketmind.experiments.figures import draw_timeline

LIAR = "n11b_tactical_analysis"
timeline = run_scenario(CLIP, 0.92, lie_once={LIAR})

first_try  = min(s for s, n in timeline if n == LIAR)
regen_step = min(s for s, n in timeline if n == "regen_prep")

fig_path = PROJECT_ROOT / "paper" / "CricketMind_ICCIT" / "figures" / "fig_execution_trace.pdf"
fig_path.parent.mkdir(parents=True, exist_ok=True)
fig = draw_timeline(
    timeline,
    hatched={(first_try, LIAR)},
    notes={first_try + 1:  "\u2717 rejects the fabricated figure",
           regen_step:     "re-runs only the failed agent",
           regen_step + 1: "second attempt",
           regen_step + 2: "\u2713 all findings verified"},
    save_to=fig_path,
)
print("saved:", fig_path.relative_to(PROJECT_ROOT), f"({fig_path.stat().st_size // 1024} KB)")

## B. Query-conditioned routing

29 prompts: for each of the twelve stakeholders a **keyword** version using the
agent's trigger words and a **plain** version avoiding all of them, plus five
tricky ones. Each is routed once by the keyword router (deterministic) and three
times by the LLM router.

The LLM answers are cached in `experiments/routing/llm_routing_runs.json` and
only re-run when the model, the run count or the prompt list changes — so this
notebook works with Ollama switched off.

In [ ]:
keyword_rows = route_all("keyword")

print(f"{'id':16s} {'target':8s} agents woken")
print("-" * 76)
for r in keyword_rows:
    print(f"{r['id']:16s} {r['target_woken'] or '':8s} "
          + ", ".join(agent_name(a) for a in r["woken"]))

from cricketmind import POSTURE_AGENTS
for r in keyword_rows:
    assert r["status"] == "ok" and 1 <= len(r["woken"]) <= MAX_ACTIVE_AGENTS
    if r["clip"] is None:
        assert not set(r["woken"]) & POSTURE_AGENTS
print("\nOK: the cap and the evidence rule held on all 29 prompts")

In [ ]:
llm_runs = load_or_run_llm(runs=3, rerun=False)
if llm_runs is None:
    print("No saved LLM answers and Ollama is not running.")
    print("Start Ollama, then:  ollama pull qwen2.5:7b")
else:
    print(f"{len(llm_runs)} runs x {len(llm_runs[0])} prompts")

In [ ]:
c = compare(keyword_rows, llm_runs)
MARK = {"signal": "+", "default": "~", "no": "x", None: " "}
print("target woken:  + the router chose it   ~ only via the default pair   x not woken")
print()
print(f"{'id':16s}   {'keyword':34s}   {'LLM (run 1)':34s} stable")
print("-" * 104)
for kw, llm, same in zip(keyword_rows, llm_runs[0], c["stable"]):
    print(f"{kw['id']:16s} {MARK[kw['target_woken']]} "
          f"{', '.join(agent_name(a) for a in kw['woken'])[:34]:34s} "
          f"{MARK[llm['target_woken']]} "
          f"{', '.join(agent_name(a) for a in llm['woken'])[:34]:34s} "
          f"{'yes' if same else 'NO'}")

In [ ]:
summary = T.write_routing_summary(c["per_run"], c["stable"], c["seconds"])
plain   = T.write_plain_table(llm_runs[0])
flow    = T.write_flow_table(record_flows(), stages=PAPER_FLOW_STAGES)
print(summary)
print(plain)
print(flow)

### One query through the graph

`stream_mode="updates"` reports what each node wrote. Two questions, the same
pipeline, different evidence and different experts.

In [ ]:
from cricketmind.experiments.routing_eval import FLOW_PROMPTS, FLOW_STAGES, describe_write

flows = record_flows()
for label, q in FLOW_PROMPTS.items():
    print(f"{label}: {q}")
for stage, nodes in FLOW_STAGES:
    print(f"\n{stage}")
    for label, writes in flows.items():
        lines = [describe_write(n, u) for n, u in writes if n in nodes]
        print(f"   {label}: {' | '.join(lines) or '-'}")

## C. Boundary of the Faith Check

The gate verifies **values, not the claims wrapped around them**. Four versions
of the Coaches agent's answer locate the line.

`Settings.template_overrides` changes what an agent *says* without changing what
it *cites* — which is exactly the distinction this result is about.

In [ ]:
r3 = run_boundary_cases()

print(f"{'case':14s} {'Faith Check':12s} {'regens':>6s}  {'reaches the user':18s} cites")
print("-" * 96)
for r in r3:
    print(f"{r['case']:14s} {'REJECTED' if r['rejected'] else 'passed':12s} "
          f"{r['regens']:6d}  {'yes' if r['claim_in_report'] else 'no':18s} {r['cites']}")

by = {r["case"]: r for r in r3}
assert not by["honest"]["rejected"]        and by["honest"]["claim_in_report"]
assert by["fabricated"]["rejected"]        and not by["fabricated"]["claim_in_report"]
assert not by["misattributed"]["rejected"] and by["misattributed"]["claim_in_report"]
assert not by["uncited"]["rejected"]       and by["uncited"]["claim_in_report"]
assert by["misattributed"]["cites"] == by["honest"]["cites"]
print()
print("OK: a fabricated citation is caught; a swapped-subject claim and an")
print("    uncited number both pass and reach the user -- the gate's boundary.")

---

## What was written

| File | Table in the paper |
|---|---|
| `tables/tab_execution_profile.tex` | I |
| `tables/tab_routing_summary.tex` | II |
| `tables/tab_routing_plain.tex` | III |
| `tables/tab_prompt_flow.tex` | IV |
| `figures/fig_execution_trace.pdf` | not used — slides and thesis |

Run `python -m pytest` from the repository root to check the same results
without starting Jupyter.